# FitSense on Colab
Runtime > Change runtime type > **T4 GPU** before running anything. Run cells top to bottom.

## 1. Setup

In [ ]:
!rm -rf fitsense && git clone -q https://github.com/Rakshit1899/fitsense.git
%cd /content/fitsense
!pip install -q -r requirements.txt
import os, torch
os.environ["PYTHONPATH"] = "src"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND, switch runtime to T4")

## 2. Optional: save results to Google Drive
Colab wipes everything if the session resets. Running this keeps a copy of the model and reports in your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/fitsense_outputs'
os.makedirs(SAVE, exist_ok=True)

## 3. Data and baseline (about 5 minutes)
The baseline is retrained here so later steps can compare both models.

In [ ]:
!python -m fitsense.data
!python -m fitsense.baseline 2>&1 | grep -A8 "baseline_test" 

## 4. Dry run (about 5 to 10 minutes)
Trains on 5,000 reviews to confirm everything works. The score will be low, that is expected.

In [ ]:
!python -m fitsense.train_transformer --sample 5000 --epochs 1 2>&1 | grep -A8 "_test:"

## 5. Full DistilBERT training (roughly 1 to 2 hours)
Keep the tab open. If you are short on time, add `--max_len 128` to roughly halve it.

In [ ]:
!python -m fitsense.train_transformer --epochs 2 2>&1 | grep -A8 "_test:"

In [ ]:
# back up the trained model right away
!cp -r models/distilbert-base-uncased/best "$SAVE/distilbert_best" 2>/dev/null && echo "saved model to Drive" 

## 6. Product-level evaluation
Does the rolled-up advice (runs small / true to size / runs large) match what customers reported, on products the model never saw?

In [ ]:
!python -m fitsense.evaluate_items 2>&1 | grep -A7 "product level"

## 7. Footwear test, part 1: get shoe reviews to label
Streams Amazon shoe reviews that mention sizing and writes 250 of them to a CSV. Takes a few minutes.

In [ ]:
!python -m fitsense.footwear sample --n 250
from google.colab import files
files.download('data/footwear_to_label.csv')

## 8. Label them (on your laptop)
Open `footwear_to_label.csv` in Excel. In the **label** column write `small`, `fit` or `large` based only on what the review says about fit. Write `skip` if it doesn't say.

Tips: "size up", "too tight", "toes hit the end" = small. "Size down", "too big", "heel slips" = large. "True to size", "fits perfectly" = fit. Aim for at least 150 usable labels.

Save as **footwear_labeled.csv** (CSV format) and upload it with the next cell.

In [ ]:
uploaded = files.upload()  # choose footwear_labeled.csv
!mv footwear_labeled.csv data/footwear_labeled.csv

## 9. Footwear test, part 2: score both models

In [ ]:
!python -m fitsense.footwear score 2>&1 | grep -E "labeled|macro F1"

## 10. Download all results

In [ ]:
!cp data/footwear_labeled.csv reports/ 2>/dev/null; zip -qr fitsense_reports.zip reports
!cp fitsense_reports.zip "$SAVE/" 2>/dev/null
files.download('fitsense_reports.zip')